# SAM 2 FULL UNFREEZE & DOMAIN ADAPTATION
**Version 2.3: The Full Cook**
- **Full Unfreeze**: Entire SAM 2 model is trainable.
- **Precision Points**: Dynamically synced scaling.
- **Hybrid Loss**: 0.2 BCE + 0.8 Dice.

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git accelerate pycocotools -q

In [ ]:
import os, cv2, json, torch, numpy as np, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from glob import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Sam2Model
from pycocotools import mask as mask_util
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Metrics

In [ ]:
def compute_iou(preds, targets):
    t = (targets > 0.5).float()
    if preds.shape[0] != t.shape[0]: t = t[:preds.shape[0]]
    p_upsampled = torch.nn.functional.interpolate(preds.unsqueeze(1), size=t.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
    p_binary = (torch.sigmoid(p_upsampled) > 0.5).float()
    intersection = (p_binary * t).sum(dim=(1, 2))
    union = (p_binary + t).clamp(0, 1).sum(dim=(1, 2))
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.medianBlur(img, 5)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

## 2. Dataset

In [ ]:
class HieroglyphsDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.data_dir = data_dir
        self.processor = processor
        self.json_paths = sorted(glob(os.path.join(data_dir, "*.json")))
        
    def __len__(self):
        return len(self.json_paths)
    
    def __getitem__(self, idx):
        json_path = self.json_paths[idx]
        img_path = json_path.replace(".json", ".jpg")
        image = preprocess_image(img_path)
        if image is None: return None
        
        with open(json_path, 'r') as f: data = json.load(f)
        bboxes, gt_masks, raw_points = [], [], []
        orig_w, orig_h = image.size
        
        for ann in data['annotations']:
            if 'segmentation' not in ann: continue
            x, y, w, h = ann['bbox']
            bboxes.append([x, y, x + w, y + h])
            raw_points.append([x + w/2, y + h/2])
            gt_masks.append((mask_util.decode(ann['segmentation']) > 0).astype(np.float32))
        
        if not bboxes: return None
        MAX = 100
        inputs = self.processor(image, input_boxes=[bboxes[:MAX]], return_tensors="pt")
        
        target_size = 1024
        scale_x, scale_y = target_size/orig_w, target_size/orig_h
        scaled_points = [[[p[0]*scale_x, p[1]*scale_y]] for p in raw_points[:MAX]]
        
        inputs["input_points"] = torch.tensor(np.array(scaled_points), dtype=torch.float32)
        inputs["input_labels"] = torch.ones((len(scaled_points), 1), dtype=torch.int)
        inputs["gt_masks"] = torch.tensor(np.array(gt_masks[:MAX]), dtype=torch.float32)
        
        return {k: v.squeeze(0) if k != "gt_masks" else v for k, v in inputs.items()}

def custom_collate(batch):
    return [b for b in batch if b is not None]

processor = AutoProcessor.from_pretrained("facebook/sam2-hiera-large")
BASE = "/kaggle/input/datasets/karimtawfik/hieroglyphics-segmentation-data-sam-2-format/segmentation.v1-2023-03-12-7-33pm.sam2/"
train_loader = DataLoader(HieroglyphsDataset(BASE+"train", processor), batch_size=1, shuffle=True, collate_fn=custom_collate)
valid_loader = DataLoader(HieroglyphsDataset(BASE+"valid", processor), batch_size=1, shuffle=False, collate_fn=custom_collate)

## 3. Training

In [ ]:
model = Sam2Model.from_pretrained("facebook/sam2-hiera-large").to(device)
for param in model.parameters(): param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
bce_loss = nn.BCEWithLogitsLoss()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        p = torch.sigmoid(pred).reshape(-1)
        t = target.reshape(-1)
        inter = (p * t).sum()
        return 1 - (2. * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)

dice_fn = DiceLoss()

def calc_loss(pred, target):
    t = (target > 0.5).float()
    if t.shape[0] > pred.shape[0]: t = t[:pred.shape[0]]
    p_up = torch.nn.functional.interpolate(pred.unsqueeze(1), size=t.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
    return 0.2 * bce_loss(p_up, t) + 0.8 * dice_fn(p_up, t)

for epoch in range(15):
    model.train()
    t_l, t_i = [], []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")
    for batch in pbar:
        if not batch: continue
        for item in batch:
            inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
            gt = item["gt_masks"].to(device)
            out = model(**inputs, multimask_output=False)
            pred = out.pred_masks.squeeze(0)
            loss = calc_loss(pred, gt)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            t_l.append(loss.item())
            t_i.append(compute_iou(pred.detach(), gt))
            pbar.set_postfix({"loss": f"{np.mean(t_l):.4f}", "iou": f"{np.mean(t_i):.4f}"})

    model.eval()
    v_i = []
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="VALID"):
            if not batch: continue
            for item in batch:
                inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
                gt = item["gt_masks"].to(device)
                out = model(**inputs, multimask_output=False)
                pred = out.pred_masks.squeeze(0)
                v_i.append(compute_iou(pred, gt))
    print(f"Summary: T-Loss: {np.mean(t_l):.4f} | T-IoU: {np.mean(t_i):.4f} | V-IoU: {np.mean(v_i):.4f}")

## 4. Visual Verification

In [ ]:
model.eval()
batch = next(iter(valid_loader))
item = batch[0]
inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
with torch.no_grad():
    out = model(**inputs, multimask_output=False)
img = item["pixel_values"].permute(1, 2, 0).cpu().numpy()
img = (img - img.min()) / (img.max() - img.min())
plt.imshow(img)
pred_masks = torch.sigmoid(out.pred_masks.squeeze(0).squeeze(1)) > 0.5
for i in range(min(5, pred_masks.shape[0])):
    m = pred_masks[i].cpu().numpy()
    plt.imshow(np.dstack([m*np.random.rand(), m*np.random.rand(), m*np.random.rand(), m*0.5]))
plt.show()